RQ1 ------------------------------------------------

Random Forest

Facemask wearing - Before mandates - covid model

Dataset with state and covid rollig cases and deaths are considered here for the analysis.

In [ ]:
# import libraries
import pandas as pd
import numpy as np

In [9]:
before_train_facemask = pd.read_csv("before_train_facemask.csv")
before_test_facemask = pd.read_csv("before_test_facemask.csv")

In [10]:
before_train_facemask.columns

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'household_size',
       'Wellbeing', 'Perceived Severity', 'Perceived Susceptibility',
       'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment'

In [11]:
# remove the target variables and identifiers
drop_cols = ['RecordNo', 'Date','face_mask_scale', 'face_mask_binary','general_protective_behavior_scale',
              'general_protective_behavior_binary','mandate_start_date']



# predictors
x_train = before_train_facemask.drop(columns=drop_cols)
x_test = before_test_facemask.drop(columns=drop_cols)

x_train = x_train.astype(float)  # converting boolean to float 
x_test = x_test.astype(float)

In [12]:
print(x_train.columns)
print(x_train.shape)

Index(['Non-household contacts', 'age', 'household_size', 'Wellbeing',
       'Perceived Severity', 'Perceived Susceptibility',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_period', 'Isolate if unwell_Not sure',
       'Isolate if unwell_Yes', 'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment's response_Don't 

In [13]:
# target variable 

y_train = before_train_facemask["face_mask_binary"]
y_test = before_test_facemask["face_mask_binary"]

Random Forest

In [ ]:
# from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib


cv = StratifiedKFold( # 5 fold cross validation
    n_splits=5,
    shuffle=True,
    random_state=42
)

  
cv_rf = Pipeline([   # standard scaler is not important in RF as RF makes decisions on the order of them not the scale
        ('ros', RandomOverSampler(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42,n_jobs=-1))
    ])   


params = { # apply the parameters to the rf step of the pipeline - rf__n_estimators  double underscore
    
        'rf__n_estimators': [250],   # number of trees   100,250
        'rf__max_depth': [5,7,10],
        'rf__min_samples_split': [2,10,20],  # 2,10
        'rf__min_samples_leaf': [1,5,10],    # 1,5
        'rf__max_features': ['sqrt','log2']  # important for RF
    }

grid = GridSearchCV( # tunes parameters
        cv_rf,
        params,
        cv=cv,
        scoring={  # evaluates all metrics
        'roc_auc': 'roc_auc',
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1'},
        refit='roc_auc',  # choose the best model
        return_train_score=False,

        # scoring='roc_auc', # for each combination, it uses 5 fold cv and calculates roc auc

        n_jobs=-1 # use all available CPU cores for parallel processing
    )

grid.fit(x_train, y_train)

best_rf = grid.best_estimator_   # best model

best_parameters = grid.best_params_   # best hyper parameters

results_rf = pd.DataFrame(grid.cv_results_)

pred = best_rf.predict(x_test)
prob = best_rf.predict_proba(x_test)[:,1]

print("Accuracy:", round(accuracy_score(y_test, pred),4))
print("ROC AUC:", round(roc_auc_score(y_test, prob),4))
print("Precision:", round(precision_score(y_test, pred),4))
print("Recall:", round(recall_score(y_test, pred),4))
print("F1:", round(f1_score(y_test, pred),4))


joblib.dump(best_rf, "facemask_before_covidModel_RF_rolling.pkl")
results_rf.to_csv("facemask_before_covidModel_RF_rolling_results.csv", index=False)

Accuracy: 0.7588
ROC AUC: 0.8379
Precision: 0.5227
Recall: 0.7359
F1: 0.6113


In [15]:
joblib.dump(best_parameters, "facemask_before_covidModel_RF_bestParameters_rolling.pkl")

['facemask_before_covidModel_RF_bestParameters_rolling.pkl']